# Strata on Kaggle — DevZoneX

Run a Strata-supported model on a Kaggle GPU, save your own private model/runtime caches, and connect OpenCode Desktop through an authenticated temporary Cloudflare Quick Tunnel.

## Before running
1. Enable Internet and an available GPU accelerator in Kaggle notebook settings.
2. Add a Kaggle API token as a Secret named `KAGGLE_API_TOKEN` under Add-ons → Secrets and enable notebook access.
3. Edit the configuration cell with your Kaggle username and a model family/variant supported by Strata.
4. For later sessions, attach your own private model and runtime datasets via Add Input.
5. Read the chosen model's license. Enable the explicit upload confirmation only if the terms permit copying to your own private Kaggle Dataset.

**Compatibility warning:** this is not a universal Hugging Face loader. Strata supports specific families/variants; run and review the launch plan before downloading large files.

In [ ]:
# ===== USER CONFIGURATION: EDIT THIS CELL =====
KAGGLE_USERNAME = "your-kaggle-username"
MODEL_FAMILY = "gyro"
MODEL_VARIANT = "gyro-s"
CONTEXT = "32k"
VISION = "off"
GPU_IDS = ""                         # blank = detect all visible NVIDIA GPUs
MODEL_DATA_DIR = "/root/strata-local"

import re
_slug = lambda value: re.sub(r"[^a-z0-9-]+", "-", value.lower()).strip("-")
MODEL_DATASET_SLUG = _slug(f"strata-model-{MODEL_FAMILY}-{MODEL_VARIANT}")
RUNTIME_DATASET_SLUG = "strata-runtime-cache"

UPLOAD_CACHES = True
I_CONFIRM_MODEL_LICENSE_AND_PRIVATE_STORAGE = False
FORCE_UPDATE_MODEL_CACHE = False
FORCE_UPDATE_RUNTIME_CACHE = False

print("Model selection:", MODEL_FAMILY, MODEL_VARIANT)
print("Model cache:", f"{KAGGLE_USERNAME}/{MODEL_DATASET_SLUG}")
print("Runtime cache:", f"{KAGGLE_USERNAME}/{RUNTIME_DATASET_SLUG}")
print("Model private-storage confirmation:", I_CONFIRM_MODEL_LICENSE_AND_PRIVATE_STORAGE)

## 1. Load helper scripts and Kaggle credentials

This clones the public repository and reads the Kaggle token from Secrets. The helper stores the token in a temporary private config path and never prints it.

In [ ]:
import os, sys, subprocess, json, re, shutil
from pathlib import Path

REPO_URL = "https://github.com/iAmAjayTeli/devzonex-strata-kaggle.git"
REPO_DIR = Path("/kaggle/working/devzonex-strata-kaggle")
if not (REPO_DIR / "scripts" / "strata_kaggle.py").is_file():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

sys.path.insert(0, str(REPO_DIR))
from scripts import strata_kaggle as sk

KAGGLE_ENV = sk.kaggle_env(KAGGLE_USERNAME, secret_name="KAGGLE_API_TOKEN")
sk.install_kaggle_cli(KAGGLE_ENV)
HARDWARE = sk.hardware_summary()
print(json.dumps(HARDWARE, indent=2))
if not HARDWARE["gpu_names"]:
    raise RuntimeError("No NVIDIA GPU detected. Enable a GPU accelerator in Kaggle settings.")


## 2. Restore your caches or install Strata

First-time run: no caches need to be attached; Strata can fetch the selected model/engine after the plan is approved.

Returning run: use Add Input to attach your own model-cache and runtime-cache datasets. Restoring the model from the read-only dataset mount to local storage may still take several minutes.

In [ ]:
MODEL_ROOT = Path(MODEL_DATA_DIR)
MODEL_ROOT.mkdir(parents=True, exist_ok=True)

RUNTIME_CACHE_RESTORED = sk.restore_runtime_cache(KAGGLE_USERNAME, RUNTIME_DATASET_SLUG)
MODEL_CACHE_RESTORED = sk.restore_model_cache(KAGGLE_USERNAME, MODEL_DATASET_SLUG, MODEL_ROOT, workers=8)
sk.install_strata_if_missing()

print("Runtime cache restored:", RUNTIME_CACHE_RESTORED)
print("Model cache restored:", MODEL_CACHE_RESTORED)
print("Strata version check complete.")

## 3. Review the Strata plan

Read the plan output carefully before proceeding. If Strata rejects the selected family/variant, update the configuration cell and rerun this cell. Do not repeatedly run an unsupported plan.

In [ ]:
if not GPU_IDS.strip():
    GPU_IDS = sk.default_gpu_list()
print("GPU selection:", GPU_IDS)
PLAN_OUTPUT = sk.run_plan(
    family=MODEL_FAMILY,
    variant=MODEL_VARIANT,
    context=CONTEXT,
    vision=VISION,
    gpu_list=GPU_IDS,
    data_dir=MODEL_ROOT,
)

## 4. Launch the authenticated Strata server and Cloudflare tunnel

The helper starts Strata with `--access cloudflare`, waits for a loaded model and API-key protection, then starts a Quick Tunnel if Strata did not create one. Quick Tunnel URLs are temporary and publicly reachable; the Strata API key must remain private.

In [ ]:
import requests

# Reuse an already-running server only if it is loaded and API-key protected.
server_is_ready = False
health_json = {}
try:
    r = requests.get("http://127.0.0.1:8080/health", timeout=5)
    health_json = r.json() if r.ok else {}
    server_is_ready = bool(health_json.get("loaded") and health_json.get("api_key"))
except Exception:
    pass

if server_is_ready:
    print("Reusing the healthy, API-key-protected Strata server.")
    LAUNCH_OUTPUT = ""
else:
    if health_json.get("loaded") and not health_json.get("api_key"):
        sk.run_command(["strata", "launch", "--stop"], timeout=60)
    LAUNCH_OUTPUT = sk.launch_strata(
        family=MODEL_FAMILY, variant=MODEL_VARIANT, context=CONTEXT,
        vision=VISION, gpu_list=GPU_IDS, data_dir=MODEL_ROOT,
    )

match = re.search(r"https://[A-Za-z0-9-]+\.trycloudflare\.com", LAUNCH_OUTPUT)
LAUNCH_TUNNEL_URL = match.group(0) if match else None
TUNNEL_URL = sk.start_quick_tunnel(LAUNCH_TUNNEL_URL)
HEALTH = sk.wait_health(timeout_seconds=900)
print("Loaded:", HEALTH.get("loaded"))
print("Model:", HEALTH.get("model"))
print("Temporary HTTPS endpoint:", TUNNEL_URL)

## 5. Verify the public API and run a short generation

The next cell checks that unauthenticated API requests are rejected with HTTP 401/403, authenticated requests can list models, and a short authenticated chat completion succeeds. It never prints the Strata API key.

In [ ]:
TEST_RESULT = sk.verify_public_endpoint(TUNNEL_URL)
MODEL_ID = TEST_RESULT["model_id"]
print("End-to-end authenticated API test: PASS")
print("Model ID for OpenCode:", MODEL_ID)

## 6. Save your personal private caches

The model and runtime are stored in separate datasets owned by your Kaggle account. Dataset upload is disabled for the model unless you explicitly confirm the model's license permits your private copy. The uploader never passes a public-visibility flag. Kaggle may need time to process datasets after the upload finishes.

In [ ]:
if UPLOAD_CACHES:
    MODEL_UPLOADED = sk.upload_model_cache(
        data_dir=MODEL_ROOT,
        username=KAGGLE_USERNAME,
        slug=MODEL_DATASET_SLUG,
        family=MODEL_FAMILY,
        variant=MODEL_VARIANT,
        env=KAGGLE_ENV,
        license_confirmed=I_CONFIRM_MODEL_LICENSE_AND_PRIVATE_STORAGE,
        force=FORCE_UPDATE_MODEL_CACHE,
    )
    RUNTIME_UPLOADED = sk.upload_runtime_cache(
        username=KAGGLE_USERNAME,
        slug=RUNTIME_DATASET_SLUG,
        env=KAGGLE_ENV,
        force=FORCE_UPDATE_RUNTIME_CACHE,
    )
    print("Model cache upload submitted:", MODEL_UPLOADED)
    print("Runtime cache upload submitted:", RUNTIME_UPLOADED)
    print("Check the Kaggle dataset pages and wait for processing to finish before ending this session.")
else:
    print("Cache uploads disabled by configuration.")

## 7. Connect OpenCode Desktop

Merge the printed provider entry into your existing OpenCode configuration; do not replace unrelated providers. Retrieve the Strata API key privately in the Kaggle shell with `strata launch --show-key`. Save it locally on your Windows PC at `~/.config/opencode/strata-key.txt`. Never commit or share the key.

Keep this Kaggle session and tunnel running while using the model. The URL may change after a restart.

In [ ]:
print("OpenCode baseURL:", TUNNEL_URL.rstrip("/") + "/v1")
print("OpenCode model ID:", MODEL_ID)
print("\nMerge this provider entry inside the existing top-level providers object:")
print(sk.opencode_config_snippet(TUNNEL_URL, MODEL_ID))
print("\nRetrieve the API key privately in the Kaggle terminal; do not copy it to notebook output:")
print("strata launch --show-key")